In [1]:
!pip install datasets
!pip install lime
!pip install joblib
!pip install transformers
!pip install torch
!pip install sentencepiece
!pip install accelerate
!pip install bitsandbytes
!pip install scipy
!pip install scikit-learn
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 14.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system ==

In [3]:
import numpy as np
import pandas as pd
import re
import logging
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from lime.lime_text import LimeTextExplainer
import joblib

# Configure logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Privacy-preserving text processing
def anonymize_text(text):
    """Remove PII from text"""
    text = re.sub(r'\S+@\S+', '[EMAIL]', text)
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{4} ?\d{4} ?\d{4} ?\d{4}\b', '[CREDIT_CARD]', text)
    return text

# Data loading and processing
def load_and_process_data():
    """Load and process customer support conversations"""
    try:
        # Load dataset from Hugging Face
        ds = load_dataset("TNE-AI/customer-support-on-twitter-conversation")

        # Process conversations with proper structure handling
        processed_data = []
        for conv in ds['train']:
            text = conv.get('conversation', '')
            label = 'other'
            if any(kw in text.lower() for kw in ['problem', 'issue', 'help']):
                label = 'support'
            elif any(kw in text.lower() for kw in ['thank', 'appreciate', 'good']):
                label = 'positive'
            processed_data.append({
                'text': anonymize_text(text),
                'label': label
            })

        df = pd.DataFrame(processed_data)
        label_counts = df['label'].value_counts()
        logging.info("Label distribution:\n%s", label_counts)
        print("Label distribution:\n", label_counts)

        min_samples = 10
        valid_labels = label_counts[label_counts >= min_samples].index
        df = df[df['label'].isin(valid_labels)]

        if df['label'].nunique() < 2:
            raise ValueError("Insufficient label diversity for classification")

        return df

    except Exception as e:
        logging.error("Error processing dataset: %s", str(e))
        raise

# Model training with validation
def train_model(X_train, y_train):
    unique_classes = np.unique(y_train)
    if len(unique_classes) < 2:
        raise ValueError(f"Need at least 2 classes, found: {unique_classes}")

    model = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        random_state=42
    )
    model.fit(X_train, y_train)
    return model

# Evaluation metrics
def evaluate_model(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_true, y_pred, average='weighted', zero_division=0)
    }

# Explainability with LIME
def explain_prediction(model, vectorizer, text):
    explainer = LimeTextExplainer(class_names=model.classes_)
    exp = explainer.explain_instance(
        text,
        lambda x: model.predict_proba(vectorizer.transform(x)),
        num_features=10
    )
    return exp.as_list()

# Main workflow
if __name__ == "__main__":
    try:
        df = load_and_process_data()
        logging.info("Final label distribution:\n%s", df['label'].value_counts())
        print("\nFinal label distribution:\n", df['label'].value_counts())

        X_train, X_test, y_train, y_test = train_test_split(
            df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
        )

        vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words='english',
            ngram_range=(1, 2)
        )
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        model = train_model(X_train_vec, y_train)

        y_pred = model.predict(X_test_vec)
        metrics = evaluate_model(y_test, y_pred)
        logging.info("Classification metrics:\n%s", pd.Series(metrics))
        print("\nClassification metrics:\n", pd.Series(metrics))

        # LIME explanation on sample text
        try:
            sample_text = X_test.iloc[0]
            explanation = explain_prediction(model, vectorizer, sample_text)
            print("\nSample text:\n", sample_text)
            print("\nLIME explanation for sample text:")
            for feature, weight in explanation:
                print(f"{feature}: {weight:.4f}")
        except Exception as e:
            logging.warning("Explanation failed: %s", str(e))
            print("Explanation failed:", str(e))

        # Save model artifacts
        joblib.dump(model, "support_classifier_model.pkl")
        joblib.dump(vectorizer, "support_classifier_vectorizer.pkl")
        print("\nModel artifacts saved successfully.")

        # ----------------------------
        # USER INPUT for live testing
        # ----------------------------
        print("\n--- Test the model with your own input ---")
        user_input = input("Enter a customer support message: ")
        if user_input.strip():
            clean_input = anonymize_text(user_input)
            vec_input = vectorizer.transform([clean_input])
            pred = model.predict(vec_input)[0]
            print(f"\nPredicted label: {pred}")

            # Optional: explain prediction
            explanation = explain_prediction(model, vectorizer, clean_input)
            print("\nLIME explanation for your input:")
            for feature, weight in explanation:
                print(f"{feature}: {weight:.4f}")
        else:
            print("No input provided.")

    except Exception as e:
        logging.error("Workflow failed: %s", str(e))
        print("Workflow failed:", str(e))


Label distribution:
 label
support     329457
other       312226
positive    152652
Name: count, dtype: int64

Final label distribution:
 label
support     329457
other       312226
positive    152652
Name: count, dtype: int64

Classification metrics:
 accuracy     0.986756
precision    0.986984
recall       0.986756
f1           0.986753
dtype: float64

Sample text:
 Customer: mfw @116136 internet goes from bad to complete shit and then somehow becomes even shitter to the point where you can't do homework. https://t.co/2Isz6A2MGZ
Support: Can you please DM us your account info so we can help with any connectivity issues?-FRL
Customer: Called support line already. Seems to be infrastructure/system issues or something in the area (according to the automated message)
Support: We are here 24/7 to help if you have any other questions or concerns. -FRL

LIME explanation for sample text:
issues: -0.0130
help: -0.0128
Called: 0.0018
point: 0.0016
Customer: 0.0014
DM: 0.0013
automated: 0.0012
